In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow 
import dagshub


In [3]:
import dagshub
dagshub.init(repo_owner='akashkangule18', repo_name='SWIGGI-DELIVERY-TIME-PREDICTION', mlflow=True)

Accessing as akashkangule18

Initialized MLflow to track repo "akashkangule18/SWIGGI-DELIVERY-TIME-PREDICTION"

Repository akashkangule18/SWIGGI-DELIVERY-TIME-PREDICTION initialized!

In [4]:
data = pd.read_csv("../data/raw/uncleaned_swiggy (1).xls")
data.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [5]:
df = data.copy()

In [6]:
# # renames the cols

df.rename(columns = {'Delivery_person_ID':'rider_id',
                     'Delivery_person_Age':'age',
                     'Delivery_person_Ratings':'rating',
                     'Restaurant_latitude':'rest_lat',
                     'Restaurant_longitude':'rest_long',
                     'Delivery_location_latitude':'destination_lat',
                     'Delivery_location_longitude':'destination_long',
                     'Order_Date':'order_date',
                     'Time_Orderd':'order_time',
                     'Time_Order_picked':'order_picked_time',
                     'Weatherconditions':'weather_cond',
                     'Road_traffic_density':'traffic',
                     'Vehicle_condition':'vehicle_cond',
                     'Type_of_order':'order_type',
                     'Type_of_vehicle':'vehicle_type',
                     'multiple_deliveries':'multiple_orders',
                     'Festival':'festival',
                     'City':'city_type',
                     'Time_taken(min)':'delivery_time'
                    }, inplace = True
         )


In [7]:
# lowering all the values  and removing al  the unwnated spaces like trailing and leading spaces in all columns
def lower(column):
    return column.lower().strip()

In [8]:
df['rider_id'] = df['rider_id'].apply(lower)
df['weather_cond'] = df['weather_cond'].apply(lower)
df['traffic'] = df['traffic'].apply(lower)
df['order_type'] = df['order_type'].apply(lower)
df['vehicle_type'] = df['vehicle_type'].apply(lower)
df['festival'] = df['festival'].apply(lower)
df['city_type'] = df['city_type'].apply(lower)

In [9]:
# fixing the nan values from the data from all the columns

# for age
df['age']= df['age'].replace('NaN ', np.nan)

# for rating
df['rating']= df['rating'].replace('NaN ', np.nan)

# for order time
df['order_time']= df['order_time'].replace('NaN ', np.nan)

# for order date
df['order_date']= df['order_date'].replace('nan', np.nan)

# for order picked time
df['order_picked_time']= df['order_picked_time'].replace('NaN ', np.nan)

# for traffic
df['traffic']= df['traffic'].replace('nan', np.nan)

# for vaehical condotion
df['vehicle_cond']= df['vehicle_cond'].replace('NaN ', np.nan)

# for order typpe
df['order_type']= df['order_type'].replace('nan', np.nan)

# for vehicle type
df['vehicle_type']= df['vehicle_type'].replace('nan', np.nan)

# for multiple orders
df['multiple_orders']= df['multiple_orders'].replace('NaN ', np.nan)

# for festivel
df['festival']= df['festival'].replace('nan', np.nan)

# for city type
df['city_type']= df['city_type'].replace('nan', np.nan)


In [10]:
# for weather column
df['weather_cond']= df['weather_cond'].str.split().str.get(1)
df['weather_cond']= df['weather_cond'].replace('nan',np.nan)

In [11]:
# for delivery time
df['delivery_time'] = df['delivery_time'].str.split().str.get(1)

In [12]:
# Changing datatypes
df['age'] = df['age'].astype('float')
df['rating'] = df['rating'].astype('float')
df['multiple_orders'] = df['multiple_orders'].astype(float)


# for datetime columns
df['order_date'] = pd.to_datetime(df['order_date'],dayfirst=True)
df['order_time'] = pd.to_datetime(df['order_time'], format='%H:%M:%S')
df['order_picked_time'] = pd.to_datetime(df['order_picked_time'], format='%H:%M:%S')

# with output column
df['delivery_time'] = df['delivery_time'].astype('int')

In [13]:
df['city_name'] = df['rider_id'].str.split('res').str.get(0)

In [14]:
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.month_name()
df['day'] = df['order_date'].dt.day_name()


In [15]:
# creating the weekend column
def  weekend(column):
    if column == 'Saturday' or column == 'Sunday':
        return 1
    else:
        return 0


df['weekend']= df['day'].apply(weekend)

In [16]:
# now extracting hours and miniuts form the order time and order picked time
# creating new column such order_picked_dur. it will describe that after how many miniuts order was picked
# and again creating time period such as morning, afternon, evening and night

df['ordered_hour'] = df['order_time'].dt.hour
df['ordered_miniut'] = df['order_time'].dt.minute
df['order_picked_time_minute'] = df['order_picked_time'].dt.minute

# new column order_picked_dura
df['order_picked_in_minute'] = df['order_picked_time_minute'] - df['ordered_miniut']


# new column time duration
def time(hour):

    if hour  < 12:
        return 'morning'
    elif hour <  17:
        return 'afternoon'
    elif hour < 21:
        return 'evening'
    else:
        return 'night'


df['time_duration'] = df['ordered_hour'].apply(time)

In [17]:
# now calculating distance between restarunts and destination form the lattiude adn longitude
# but before that removin nan values from this columns for the clean approach

new_df = df.dropna(subset =['rest_lat','rest_long','destination_lat','destination_long'])

In [18]:
# new_column such as distance

def haversine(lat1, lon1, lat2, lon2):
    # Earth radius in km
    R = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

new_df = df.dropna(subset=[
    'rest_lat','rest_long','destination_lat','destination_long'
]).copy()

new_df['distance_km'] = haversine(
    new_df['rest_lat'],
    new_df['rest_long'],
    new_df['destination_lat'],
    new_df['destination_long']
).round(2)

In [19]:
# # now again cleanig the dataset
# # removing unwnated cols
new_df.drop(columns=['rest_lat','rest_long','destination_lat','destination_long','order_date','order_time','order_picked_time', 'year','order_picked_time_minute'], inplace = True)

In [20]:
# and again therr are some negative values in order_pick in minute so fixing the negative values
new_df['order_picked_in_minute'] = np.where(
    new_df['order_picked_in_minute'] < 0,
    new_df['order_picked_in_minute'] + 60,
    new_df['order_picked_in_minute']
)


# again need to convert all the categorical values into lower
new_df['month_name'] = new_df['month_name'].apply(lower)
new_df['day'] = new_df['day'].apply(lower)

In [21]:
# again creating new coulumn as distnce route
def route(distance):
    if distance < 5:
        return 'short'
    elif distance > 5 and distance < 10:
        return 'medium'
    else :
        return 'long'

new_df['distance_route']  = new_df['distance_km'].apply(route)

In [22]:
new_df.head()

,ID,rider_id,age,rating,weather_cond,traffic,vehicle_cond,order_type,vehicle_type,multiple_orders,...,month,month_name,day,weekend,ordered_hour,ordered_miniut,order_picked_in_minute,time_duration,distance_km,distance_route
0,0x4607,indores13del02,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,...,3,march,saturday,1,11.0,30.0,15.0,morning,3.03,short
1,0xb379,bangres18del02,34.0,4.5,stormy,jam,2,snack,scooter,1.0,...,3,march,friday,0,19.0,45.0,5.0,evening,20.18,long
2,0x5d6d,bangres19del01,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,...,3,march,saturday,1,8.0,30.0,15.0,morning,1.55,short
3,0x7a6a,coimbres13del02,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,...,4,april,tuesday,0,18.0,0.0,10.0,evening,7.79,medium
4,0x70a2,chenres12del01,32.0,4.6,cloudy,high,1,snack,scooter,1.0,...,3,march,saturday,1,13.0,30.0,15.0,afternoon,6.21,medium


In [23]:
# dropping mentioned columns
new_df.drop(columns =['time_duration','ordered_miniut','order_picked_in_minute','month_name','day','order_type','distance_route'],inplace = True)

In [24]:
# checking for null values
new_df.isnull().sum()

ID                    0
rider_id              0
age                1854
rating             1908
weather_cond        616
traffic             601
vehicle_cond          0
vehicle_type          0
multiple_orders     993
festival            228
city_type          1200
delivery_time         0
city_name             0
month                 0
weekend               0
ordered_hour       1731
distance_km           0
dtype: int64

In [ ]:
# dropping the null values beacuse we ran two experiments and experiment with droppping null values ig giving the better results as compare to 
# null values imputation
new_df.dropna(inplace = True)

In [26]:
new_df.drop(columns = ['rider_id','ID'], inplace=True)

In [27]:
new_df.isnull().sum()

age                0
rating             0
weather_cond       0
traffic            0
vehicle_cond       0
vehicle_type       0
multiple_orders    0
festival           0
city_type          0
delivery_time      0
city_name          0
month              0
weekend            0
ordered_hour       0
distance_km        0
dtype: int64

In [51]:
# seperating data 
X = new_df.drop(columns="delivery_time")
y = new_df['delivery_time']

# spliting data
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

# seperating numerical and categorical columns
num_cols = X.select_dtypes(include=['int64','float64','int32']).columns
ohe_cols = ['weather_cond', 'vehicle_type', 'festival', 'city_type','city_name']
ord_cols = ['traffic']

# getting necessary libraries
from sklearn.preprocessing import RobustScaler, PowerTransformer,OrdinalEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_error


# ordinal encoder
ord_encoder = OrdinalEncoder(
    categories=[['low','jam','medium','high']],
    handle_unknown= "use_encoded_value",
    unknown_value= -1
)

# buiilding pipeline
num_pipe = Pipeline([
    ('transformer',PowerTransformer(method="yeo-johnson")),
    ('scaler',RobustScaler())
])

ord_pipe = Pipeline([
    ('ord_enc',ord_encoder)
])

ohe_pipe = Pipeline([
    ('ohe_encoder',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False))
])

transformer = ColumnTransformer([
    ('num',num_pipe,num_cols),
    ('ord',ord_pipe,ord_cols),
    ('ohe',ohe_pipe,ohe_cols)
])

pipeline = Pipeline([
    ('preprocess',transformer),
    ('regressor',XGBRegressor())
])


pipeline.fit(X_train,y_train)




,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['age','rating','weather_cond',...,'weekend','ordered_hour','distance_km']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset 

In [52]:
# mkaing clone of this pipeline as base pipeline
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
base_pipeline = clone(pipeline)

base_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.

In [61]:

import warnings
warnings.filterwarnings('ignore')

In [ ]:
mlflow.set_experiment('Exp-3 comparing multiple regressor models with hyperparameters tunning')

with mlflow.start_run(run_name = "regressor models with hyperparameters tunning"):

        # using optuna

        import optuna

        def objective(trial):
            model_name = trial.suggest_categorical('model',['RFR','DTR','XGBR','LGBMR'])

            # runnig if else conditions
            if model_name == 'RFR':
                model = RandomForestRegressor(
                    n_estimators= trial.suggest_int('n_estimators',100,300),
                    max_depth= trial.suggest_int('max_depth',3,7),
                    min_samples_split= trial.suggest_int('min_samples_split',2,5),
                    min_samples_leaf= trial.suggest_int('min_sample_leaf',10,50),
                    min_impurity_decrease= trial.suggest_float('min_impurity_decrease',0.01,0.1),
                    n_jobs=-1
                )


            elif model_name == 'DTR':
                model = DecisionTreeRegressor(
                    max_depth= trial.suggest_int('max_depth',2,7),
                    min_samples_split= trial.suggest_int('min_samples_split',2,5),
                    min_samples_leaf= trial.suggest_int('min_samples_float',10,50),
                    min_impurity_decrease= trial.suggest_float('min_impurity_decrease',0.01,0.1)
                )

            elif model_name == 'XGBR':
                model = XGBRegressor(
                    n_estimators = trial.suggest_int('n_estimators',100,300),
                    max_depth = trial.suggest_int('max_depth',2,7),
                    learning_rate = trial.suggest_float('learning_rate',0.05,0.1),
                    min_child_weight = trial.suggest_int('min_child_weight',3,10),
                    subsample = trial.suggest_float('sub_sample',0.5,1),
                    colsample_bytree = trial.suggest_float('colsample_bytree',0.5,1),
                    gamma = trial.suggest_float('gamma',0.1,0.5)
                )

            else:
                model = LGBMRegressor(
                    num_leaves=trial.suggest_int('num_leaves', 20, 100),
                    min_child_samples = trial.suggest_int('min_child_samples',10, 50),
                    max_depth=trial.suggest_int('max_depth', 1, 15),
                    learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1),
                    n_estimators=500,
                    subsample=trial.suggest_float('subsample', 0.5, 1.0),
                    colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
                    reg_alpha=trial.suggest_float('reg_alpha', 0.0, 1.0),
                    reg_lambda=trial.suggest_float('reg_lambda', 0.0, 1.0),
                    random_state=42,
                    n_jobs=-1,
                    verbose = -1
                )


            # again clome the base pipelinne
            new_pipe = clone(base_pipeline)
            new_pipe.set_params(
                regressor = model
            )

            
            scores = cross_val_score(
                new_pipe,
                X_train,
                y_train,
                cv = 5,
                scoring= 'neg_mean_absolute_error'
            )

            return -scores.mean()
        
        
        # creating optuna study
        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=30)       

        # fetching best parametrs and its value
        print('best_params: ', study.best_params)
        print('best value: ', study.best_value)

        best_params = study.best_params.copy()
        best_params.pop("model")

        # building the final model
        final_model = LGBMRegressor(
            **best_params,
            n_estimators=500,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

        # cloning the base pipeline
        LGRregressor = clone(base_pipeline)
        LGRregressor.set_params(
             regressor = final_model
        )

        LGRregressor.fit(X_train,y_train)

        y_train_pred = LGRregressor.predict(X_train)
        y_test_pred = LGRregressor.predict(X_test)

        # fetching the results
        train_mae = mean_absolute_error(y_train,y_train_pred)
        test_mae = mean_absolute_error(y_test,y_test_pred)
        train_r2 = r2_score(y_train,y_train_pred)
        test_r2 = r2_score(y_test,y_test_pred)

        # logging optuna parameters
        mlflow.log_param("n_trials", 30)
        mlflow.log_params(study.best_params)
        mlflow.log_param('optuna_best_trial',study.best_trial.number)


        # logging metrics
        mlflow.log_metric('train_mae',train_mae)
        mlflow.log_metric('optuna_best_mae',study.best_value)
        mlflow.log_metric('train_r2',train_r2)
        mlflow.log_metric('test_mae',test_mae)
        mlflow.log_metric('test_r2',test_r2)

        # logging file
        mlflow.log_artifact("Swiggy-Exp3.ipynb")

        # training data
        train_df = X_train.copy()
        train_df['delivery_time'] = y_train
        train_data = train_df
        train_data = mlflow.data.from_pandas(train_data)

        # testing data
        test_df = X_test.copy()
        test_df['delivery_time'] = y_test
        test_data = test_df
        test_data = mlflow.data.from_pandas(test_data)

        # logging data
        mlflow.log_input(train_data, context= "training dataset")
        mlflow.log_input(test_data, context= 'testing dataset')

        # signature
        signature = mlflow.models.infer_signature(X_train,LGRregressor.predict(X_train))

        # log model
        mlflow.sklearn.log_model(
             sk_model=LGRregressor,
             name='LightgbmRegressor',
             signature= signature,
             serialization_format= mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE
        )



        print("training mean absolute error: ", train_mae)
        print("testing mean absolute error: ",test_mae)
        print("training r2_score: ",train_r2)
        print('testing r2_score', test_r2)

2026/06/26 17:34:49 INFO mlflow.tracking.fluent: Experiment with name 'Exp-3 comparing multiple regressor models with hyperparameters tunning' does not exist. Creating a new experiment.
[I 2026-06-26 17:34:51,531] A new study created in memory with name: no-name-55bc4dbe-69b8-4356-b4f3-0e6a3277d468
[I 2026-06-26 17:34:59,883] Trial 0 finished with value: 5.3682190758184705 and parameters: {'model': 'RFR', 'n_estimators': 180, 'max_depth': 3, 'min_samples_split': 4, 'min_sample_leaf': 33, 'min_impurity_decrease': 0.08652687584646365}. Best is trial 0 with value: 5.3682190758184705.
[I 2026-06-26 17:35:03,568] Trial 1 finished with value: 5.384838771023379 and parameters: {'model': 'DTR', 'max_depth': 3, 'min_samples_split': 3, 'min_samples_float': 10, 'min_impurity_decrease': 0.0824313550289686}. Best is trial 0 with value: 5.3682190758184705.
[I 2026-06-26 17:35:12,753] Trial 2 finished with value: 4.507897588897153 and parameters: {'model': 'RFR', 'n_estimators': 225, 'max_depth': 5, 

best_params:  {'model': 'LGBMR', 'num_leaves': 80, 'min_child_samples': 50, 'max_depth': 15, 'learning_rate': 0.023519518787827508, 'subsample': 0.8936423248850214, 'colsample_bytree': 0.976499598546252, 'reg_alpha': 0.25899146208610646, 'reg_lambda': 0.4573958514937803}
best value:  3.0179455374266917


2026/06/26 17:38:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


training mean absolute error:  2.766272790879201
testing mean absolute error:  3.0473665143248385
training r2_score:  0.8656284357744775
testing r2_score 0.8363944578500242
🏃 View run regressor models with hyperparameters tunning at: https://dagshub.com/akashkangule18/SWIGGI-DELIVERY-TIME-PREDICTION.mlflow/#/experiments/7/runs/8d5d34f3714c4ea2845959e7b3f712dd
🧪 View experiment at: https://dagshub.com/akashkangule18/SWIGGI-DELIVERY-TIME-PREDICTION.mlflow/#/experiments/7


In [70]:
print(study.best_params)
print(final_model.get_params())

{'model': 'LGBMR', 'num_leaves': 57, 'min_child_samples': 37, 'max_depth': 12, 'learning_rate': 0.017373774402495844, 'subsample': 0.9628514766056495, 'colsample_bytree': 0.8712435268087318, 'reg_alpha': 0.21226851924030313, 'reg_lambda': 0.767911464391989}
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.9490736779783902, 'importance_type': 'split', 'learning_rate': 0.012231948787586006, 'max_depth': 15, 'min_child_samples': 10, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 100, 'objective': None, 'random_state': None, 'reg_alpha': 0.9659772138867988, 'reg_lambda': 0.009653096582090622, 'subsample': 0.5052035303001198, 'subsample_for_bin': 200000, 'subsample_freq': 0}


In [69]:
scores = cross_val_score(
    LGRregressor,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)

print(-scores.mean())

3.746923263629184
